# ForgeLM — KTO Binary Feedback Alignment

Align a model using simple thumbs-up/thumbs-down feedback — no paired preferences needed.

**Requirements:** GPU runtime required (Runtime → Change runtime type → T4 GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemililik/ForgeLM/blob/main/notebooks/kto_binary_feedback.ipynb)

In [ ]:
# Step 1: Install ForgeLM (with bitsandbytes for 4-bit quantization)
!pip install -q --no-cache-dir git+https://github.com/cemililik/ForgeLM.git bitsandbytes
!pip uninstall -y wandb -q
!forgelm --version

In [ ]:
# Step 2: Check GPU
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 1)
    print(f"GPU: {gpu_name} ({vram_gb} GB VRAM)")
else:
    print("WARNING: No GPU. Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# Step 3: Create KTO dataset
# KTO format: prompt + completion + label (boolean: true=good, false=bad)
import json

kto_data = [
    {
        "prompt": "What is Python?",
        "completion": "Python is a versatile, high-level programming language known for its readable syntax and extensive library ecosystem.",
        "label": True,
    },
    {"prompt": "What is Python?", "completion": "Python is a snake.", "label": False},
    {
        "prompt": "Explain recursion.",
        "completion": "Recursion is a technique where a function calls itself to solve smaller subproblems. Each call reduces the problem size until it reaches a base case.",
        "label": True,
    },
    {"prompt": "Explain recursion.", "completion": "I don't know.", "label": False},
    {
        "prompt": "What is a database?",
        "completion": "A database is an organized collection of structured data, typically stored electronically. Common types include relational (SQL) and document (NoSQL) databases.",
        "label": True,
    },
    {"prompt": "What is a database?", "completion": "A database is a type of computer.", "label": False},
    {
        "prompt": "How does the internet work?",
        "completion": "The internet is a global network of interconnected computers that communicate using standardized protocols like TCP/IP.",
        "label": True,
    },
    {"prompt": "How does the internet work?", "completion": "Magic.", "label": False},
]

with open("kto_data.jsonl", "w") as f:
    for item in kto_data:
        f.write(json.dumps(item) + "\n")

pos = sum(1 for d in kto_data if d["label"])
neg = len(kto_data) - pos
print(f"Created {len(kto_data)} KTO examples ({pos} positive, {neg} negative)")

In [ ]:
# Step 4: Create config
config_yaml = """
model:
  name_or_path: "HuggingFaceTB/SmolLM2-135M-Instruct"
  max_length: 1024
  load_in_4bit: true

lora:
  r: 16
  alpha: 32
  target_modules: ["q_proj", "v_proj"]

training:
  trainer_type: "kto"
  kto_beta: 0.1
  output_dir: "./kto_checkpoints"
  max_steps: 50
  per_device_train_batch_size: 4
  learning_rate: 5.0e-6

data:
  dataset_name_or_path: "kto_data.jsonl"
"""

with open("kto_config.yaml", "w") as f:
    f.write(config_yaml)
print("KTO config saved! (max_steps: 50)")

In [ ]:
# Step 5: Validate
!forgelm --config kto_config.yaml --dry-run

In [ ]:
# Step 6: Train
!forgelm --config kto_config.yaml

In [ ]:
# Step 7: Verify output
import os

model_path = "./kto_checkpoints/final_model"
if not os.path.exists(model_path):
    print(f"ERROR: '{model_path}' not found. Check training output above.")
elif not os.path.isfile(os.path.join(model_path, "adapter_config.json")):
    print(f"ERROR: adapter_config.json not found in '{model_path}'.")
else:
    print(f"Training completed! Model saved to: {model_path}")
    print(f"Files: {os.listdir(model_path)}")

In [ ]:
# Step 8: Compare base model vs KTO-aligned model
import os

from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

model_path = "./kto_checkpoints/final_model"
base_model_name = "HuggingFaceTB/SmolLM2-135M-Instruct"

if not os.path.exists(model_path) or not os.path.isfile(os.path.join(model_path, "adapter_config.json")):
    print("Error: Model not found. Ensure training completed successfully.")
else:
    print("Loading base model...")
    base_model = AutoModelForCausalLM.from_pretrained(base_model_name)
    tokenizer = AutoTokenizer.from_pretrained(base_model_name)

    print("Loading KTO-aligned model...")
    ft_model = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(base_model_name), model_path)
    ft_tokenizer = AutoTokenizer.from_pretrained(model_path)

    # Mix of seen (training) and unseen prompts to test both recall and generalization
    test_prompts = [
        "What is Python?",  # seen in training
        "What is an API?",  # unseen — tests generalization
        "How do neural networks learn?",  # unseen — tests detailed response preference
    ]

    for prompt in test_prompts:
        messages = [{"role": "user", "content": prompt}]
        formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(formatted, return_tensors="pt")

        print(f"\n{'=' * 60}")
        print(f"PROMPT: {prompt}")
        print(f"{'=' * 60}")

        base_out = base_model.generate(**inputs, max_new_tokens=200, do_sample=True, temperature=0.7)
        base_resp = tokenizer.decode(base_out[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True).strip()
        print(f"\n[BASE MODEL]:\n{base_resp[:400]}")

        ft_inputs = ft_tokenizer(formatted, return_tensors="pt")
        ft_out = ft_model.generate(**ft_inputs, max_new_tokens=200, do_sample=True, temperature=0.7)
        ft_resp = ft_tokenizer.decode(ft_out[0][ft_inputs["input_ids"].shape[1] :], skip_special_tokens=True).strip()
        print(f"\n[KTO-ALIGNED]:\n{ft_resp[:400]}")

    print(f"\n{'=' * 60}")
    print("KTO-aligned model should prefer detailed, informative answers")
    print("and avoid dismissive or incorrect responses.")